# Regime Occurrence Maps

Maps the percentage of compound cold-dry (CD) and cold-wet (CW) event days
associated with each of eight weather regimes.

Regimes: NAO Negative, NAO Positive, Northwesterly, Southwesterly,
Scandinavian High, High Pressure over UK, Low close to UK, Azores High.

**Prerequisites:** `map_of_occurrence.py` (and CW equivalent) must have been run.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from netCDF4 import Dataset

# ── User configuration ──────────────────────────────────────────────────────
data_dir     = "output/arrays"
haduk_sample = ("/path/to/HadUK-Grid/5km/tasmin/day/latest/"
                "tasmin_hadukgrid_uk_5km_day_19600101-19600131.nc")
fig_dir      = "figures"
os.makedirs(fig_dir, exist_ok=True)

REGIME_LABELS = {
    "NAOneg":        "NAO Negative",
    "NAOpos":        "NAO Positive",
    "northwesterly": "Northwesterly",
    "southwesterly": "Southwesterly",
    "scandi":        "Scandinavian High",
    "highP":         "High Pressure Over UK",
    "lowUK":         "Low Close to UK",
    "azores":        "Azores High",
}
REGIME_KEYS   = list(REGIME_LABELS.keys())
PANEL_LETTERS = list("ABCDEFGHIJKLMNOP")


## 1. Load data

In [ ]:
ds  = Dataset(haduk_sample, mode="r")
lat = ds.variables["latitude"][:]
lon = ds.variables["longitude"][:]
ds.close()

def load_regime_arrays(event_type):
    """Load all regime-partitioned count arrays for CD or CW events."""
    sfx = "30yr_newP_filtered"
    names = {
        **{k: f"{event_type}_{k}_{sfx}.npy" for k in REGIME_KEYS},
        "total": (f"CD_total_recent30_newP_filtered.npy"
                  if event_type == "CD"
                  else f"CW_total_{sfx}.npy"),
    }
    out = {}
    for key, fname in names.items():
        arr = np.load(os.path.join(data_dir, fname)).astype(float)
        arr[arr < 1] = np.nan
        out[key] = arr
    return out

CD = load_regime_arrays("CD")
CW = load_regime_arrays("CW")
print("Arrays loaded.")


## 2. Colourmap

In [ ]:
colors = ["#777484", "#4848EB", "#935693", "#B885E6",
          "#E4E4FF", "#257E77", "#73E2EA"]
bins   = [1, 10, 20, 30, 40, 50, 60, 70]
cmap   = mpl.colors.ListedColormap(colors)


## 3. CD regime maps

In [ ]:
fig, ax = plt.subplots(
    figsize=(13, 7), nrows=2, ncols=4,
    subplot_kw={"projection": ccrs.PlateCarree()}
)
fig.suptitle("Percentage of CD Days by Regime", fontsize=16, y=0.95)

for idx, key in enumerate(REGIME_KEYS):
    row, col = divmod(idx, 4)
    pcm = ax[row, col].pcolormesh(
        lon, lat, (CD[key] / CD["total"]) * 100, vmin=1, vmax=70, cmap=cmap
    )
    ax[row, col].set_extent([-11, 2, 49, 62])
    ax[row, col].add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax[row, col].text(-10.5, 61, PANEL_LETTERS[idx],
                      transform=ccrs.PlateCarree(), fontsize=14, fontweight="bold")

fig.subplots_adjust(wspace=0.1, hspace=0.1)
fig.colorbar(pcm, ax=ax, cmap=cmap, ticks=[1,10,20,30,40,50,60], boundaries=bins,
             label="Percentage of CD days by regime",
             orientation="horizontal", fraction=0.046, pad=0.04, extend="max")
fig.savefig(os.path.join(fig_dir, "CD_regime_percentage_maps.png"), dpi=300, facecolor="white")
plt.show()


## 4. CW regime maps

In [ ]:
fig, ax = plt.subplots(
    figsize=(16, 7), nrows=2, ncols=4,
    subplot_kw={"projection": ccrs.PlateCarree()}
)
fig.suptitle("Percentage of CW Days by Regime", fontsize=16, y=0.95)

for idx, (key, label) in enumerate(REGIME_LABELS.items()):
    row, col = divmod(idx, 4)
    pcm = ax[row, col].pcolormesh(
        lon, lat, (CW[key] / CW["total"]) * 100, vmin=1, vmax=70, cmap=cmap
    )
    ax[row, col].set_title(label)
    ax[row, col].set_extent([-11, 2, 49, 62])
    ax[row, col].add_feature(cfeature.COASTLINE, linewidth=0.5)

fig.colorbar(pcm, ax=ax, ticks=[1,10,20,30,40,50,60], boundaries=bins,
             label="Percentage of CW days by regime",
             orientation="horizontal", fraction=0.046, pad=0.04,
             extend="max", extendfrac=0.05)
fig.savefig(os.path.join(fig_dir, "CW_regime_percentage_maps.png"), dpi=300, facecolor="white")
plt.show()


## 5. Combined CD + CW 16-panel figure

In [ ]:
fig, ax = plt.subplots(
    figsize=(11, 14), nrows=4, ncols=4,
    subplot_kw={"projection": ccrs.PlateCarree()}
)
for event_data, row_offset in [(CD, 0), (CW, 2)]:
    for idx, key in enumerate(REGIME_KEYS):
        row = row_offset + idx // 4
        col = idx % 4
        pcm = ax[row, col].pcolormesh(
            lon, lat, (event_data[key] / event_data["total"]) * 100,
            vmin=1, vmax=70, cmap=cmap
        )
        ax[row, col].set_extent([-11, 2, 49, 62])
        ax[row, col].add_feature(cfeature.COASTLINE, linewidth=0.5)
        ax[row, col].text(-10.5, 61, PANEL_LETTERS[row_offset * 4 + idx],
                          transform=ccrs.PlateCarree(), fontsize=14, fontweight="bold")

fig.subplots_adjust(left=0.05, right=0.95, top=0.89, bottom=0.11, wspace=0, hspace=0)
fig.colorbar(pcm, ax=ax, cmap=cmap, ticks=[1,10,20,30,40,50,60], boundaries=bins,
             label="Percentage of compound cold days by weather regime",
             orientation="horizontal", fraction=0.046, pad=0.04,
             extend="max", shrink=0.5)
fig.savefig(os.path.join(fig_dir, "CD_CW_regime_combined_maps.png"), dpi=300, facecolor="white")
plt.show()
